# Exploration Interactive des Données de Mobilité Bordeaux

## Projet Data Visualisation #2

Ce notebook permet d'explorer de manière interactive les données GTFS de TBM (Transports Bordeaux Métropole).

## 1. Import des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("✓ Bibliothèques importées avec succès!")

## 2. Chargement des données GTFS

In [ ]:
# Charger les données (chemins corrigés)
print("Chargement des données...")

data_path = '../Data/'  # Chemin relatif depuis notebooks/

agency = pd.read_csv(data_path + 'agency.txt')
stops = pd.read_csv(data_path + 'stops.txt')
routes = pd.read_csv(data_path + 'routes.txt')
trips = pd.read_csv(data_path + 'trips.txt')
calendar = pd.read_csv(data_path + 'calendar.txt')

print("Chargement de stop_times.txt (peut prendre 30-60 secondes)...")
stop_times = pd.read_csv(data_path + 'stop_times.txt', low_memory=False)

print(f"\n✓ Données chargées:")
print(f"  - {len(stops):,} arrêts")
print(f"  - {len(routes):,} lignes")
print(f"  - {len(trips):,} trajets")
print(f"  - {len(stop_times):,} horaires d'arrêt")

## 3. Aperçu des données

In [ ]:
# Informations sur l'agence
print("=" * 60)
print("AGENCE DE TRANSPORT")
print("=" * 60)
display(agency)

In [ ]:
# Aperçu des arrêts
print("\nÉchantillon des arrêts:")
display(stops.head(10))

In [ ]:
# Aperçu des lignes
print("\nÉchantillon des lignes:")
display(routes.head(10))

In [ ]:
# Aperçu des horaires
print("\nÉchantillon des horaires:")
display(stop_times.head(10))

## 4. Prétraitement des données

In [ ]:
# Conversion des horaires
print("Conversion des horaires...")
stop_times['arrival_time'] = pd.to_timedelta(stop_times['arrival_time'])
stop_times['departure_time'] = pd.to_timedelta(stop_times['departure_time'])

# Extraction de l'heure
stop_times['hour'] = stop_times['arrival_time'].dt.components['hours'] % 24

# IMPORTANT: Convertir les IDs en string pour éviter les erreurs de merge
print("Conversion des IDs...")
stop_times['stop_id'] = stop_times['stop_id'].astype(str)
stop_times['trip_id'] = stop_times['trip_id'].astype(str)
stops['stop_id'] = stops['stop_id'].astype(str)
trips['trip_id'] = trips['trip_id'].astype(str)
trips['route_id'] = trips['route_id'].astype(str)
routes['route_id'] = routes['route_id'].astype(str)

# Fusion des données
print("Fusion des tables...")
stop_times = stop_times.merge(
    stops[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']], 
    on='stop_id', 
    how='left'
)

stop_times = stop_times.merge(
    trips[['trip_id', 'route_id']], 
    on='trip_id', 
    how='left'
)

stop_times = stop_times.merge(
    routes[['route_id', 'route_short_name', 'route_long_name']], 
    on='route_id', 
    how='left'
)

print(f"✓ Prétraitement terminé! {len(stop_times):,} passages analysables")

## 5. Analyse exploratoire

### 5.1 Distribution horaire

In [ ]:
# Fréquentation par heure
freq_heure = stop_times.groupby('hour').size()

plt.figure(figsize=(14, 6))
plt.plot(freq_heure.index, freq_heure.values, marker='o', linewidth=2, markersize=8)
plt.fill_between(freq_heure.index, freq_heure.values, alpha=0.3)
plt.xlabel('Heure de la journée', fontsize=12)
plt.ylabel('Nombre de passages', fontsize=12)
plt.title('Distribution de la fréquentation par heure', fontsize=14, fontweight='bold')
plt.xticks(range(24), [f'{h}h' for h in range(24)], rotation=45)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nHeure de pointe: {freq_heure.idxmax()}h avec {freq_heure.max():,} passages")
print(f"Heure creuse: {freq_heure.idxmin()}h avec {freq_heure.min():,} passages")

### 5.2 Lignes les plus fréquentées

In [ ]:
# Top 20 lignes
freq_ligne = stop_times.groupby('route_short_name').size().sort_values(ascending=False).head(20)

plt.figure(figsize=(12, 8))
freq_ligne.plot(kind='barh', color='steelblue')
plt.xlabel('Nombre de passages', fontsize=12)
plt.ylabel('Ligne', fontsize=12)
plt.title('Top 20 des lignes les plus fréquentées', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTop 5 lignes:")
for ligne, count in freq_ligne.head(5).items():
    print(f"  • Ligne {ligne}: {count:,} passages")

### 5.3 Arrêts les plus fréquentés

In [ ]:
# Top arrêts
freq_arret = stop_times.groupby('stop_name').size().sort_values(ascending=False).head(20)

plt.figure(figsize=(12, 8))
freq_arret.plot(kind='barh', color='coral')
plt.xlabel('Nombre de passages', fontsize=12)
plt.ylabel('Arrêt', fontsize=12)
plt.title('Top 20 des arrêts les plus fréquentés', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTop 5 arrêts:")
for arret, count in freq_arret.head(5).items():
    print(f"  • {arret}: {count:,} passages")

## 6. Visualisation géographique

In [ ]:
# Créer une carte des arrêts
freq_arret_geo = stop_times.groupby(['stop_name', 'stop_lat', 'stop_lon']).size().reset_index()
freq_arret_geo.columns = ['stop_name', 'stop_lat', 'stop_lon', 'frequentation']
freq_arret_geo = freq_arret_geo.dropna()

# Centre de Bordeaux
center_lat = freq_arret_geo['stop_lat'].mean()
center_lon = freq_arret_geo['stop_lon'].mean()

# Créer la carte
m = folium.Map(location=[center_lat, center_lon], zoom_start=12)

# Ajouter heatmap
heat_data = [[row['stop_lat'], row['stop_lon'], row['frequentation']] 
             for idx, row in freq_arret_geo.iterrows()]
HeatMap(heat_data, radius=15, blur=25).add_to(m)

# Afficher
m

## 7. Graphique interactif avec Plotly

In [ ]:
# Créer un graphique interactif
freq_heure_df = stop_times.groupby('hour').size().reset_index()
freq_heure_df.columns = ['hour', 'count']

fig = px.line(freq_heure_df, x='hour', y='count', 
              title='Fréquentation interactive par heure',
              labels={'hour': 'Heure', 'count': 'Nombre de passages'},
              markers=True)

fig.update_traces(line_color='royalblue', line_width=3, marker_size=10)
fig.update_layout(template='plotly_white', height=500)
fig.show()

## 8. Analyses personnalisées

Utilisez les cellules ci-dessous pour faire vos propres analyses!

In [ ]:
# Votre code d'analyse ici


## Conclusion

Ce notebook vous permet d'explorer les données de mobilité de Bordeaux de manière interactive.

Pour une analyse complète automatisée, exécutez le script depuis la racine du projet:
```bash
cd ..
python main.py
```

Résultats disponibles dans:
- `visualisations/` - Tous les graphiques et cartes
- `RAPPORT_FINAL.md` - Rapport complet
- `recommandations.txt` - Recommandations d'optimisation